# ITMS M8 — fine-tune IISc's Indian-traffic detector (transfer learning), version 2

Starts from IISc's UVH-26 YOLO11-S (trained on Bengaluru CCTV) and teaches it the six vehicle types the
traffic controller counts: **car, auto-rickshaw, two-wheeler, bus, truck, bicycle**.

**How to run**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**. Allow Google Drive when asked: progress is saved there, so a disconnect loses nothing.
3. Keep this tab open. The 640 px model takes about 1–1.5 h, the 960 px one about 2–3 h. If Colab disconnects,
   reconnect and **Run all** again: it continues where it stopped.
4. After each model, `itms_uvh26_ft6v2_results.zip` in Drive (folder `itms_uvh26_finetune`) is brought up to date;
   at the end it also downloads. Put its contents in `itms/data/uvh26/finetune_v2/` and tell Claude.
   If the 960 px run cannot finish, the zip already holds everything for 640.

**What changed since version 1, and why.** Version 1's losses were `nan` in every epoch. The cause is in IISc's model
file: some of its internal normalisation statistics (BatchNorm variances, up to about 25 million) are larger than
half precision can hold (65,504). Ultralytics saves models in half precision, so those numbers were stored as
infinity; and with mixed-precision training some layer outputs overflowed too. Version 2
- re-estimates those statistics in full precision from training images before training (the control model
  `iisc_bnfix_*` is exactly that, with nothing else changed);
- trains in full precision (`amp=False`), and stops with a message if a loss is ever not a number;
- keeps a full-precision copy of the best model during training and exports it directly, never through a
  half-precision file;
- checks every exported model against the model it came from.

**What it guards against** (as version 1)
- The 250 frames used to *test* detectors are never trained on, nor are other frames that look like the same camera.
- Every IISc label is mapped to one of the six types (MUV, van → car; mini-bus, tempo-traveller → bus; LCV → truck; "Others" dropped).
  An unknown label stops the run instead of being silently dropped.
- Boxes are clipped to the image; empty, crowd and duplicate boxes are dropped; images that do not open or whose size
  disagrees with the label file are skipped. Every skip is counted in `dataset_manifest.json`.
- A separate hold-out set (300 training-split frames) decides when training stops; the test frames never influence anything.
- Fixed seeds, pinned library version, the exact image lists and the model's checksum are recorded.
- The dataset version 1 prepared in Drive is reused as it is (checked against the settings below).

In [ ]:
# ---- Settings ----
N_TRAIN = 3000          # training images
N_HOLDOUT = 300         # frames set aside to decide when to stop (from the training split, never the test frames)
SIZES = [640, 960]      # one run per input size; 640 is the CPU-friendly one and runs first
EPOCHS = 25
PATIENCE = 8            # stop a run when the hold-out score has not improved for this many epochs
BATCH = {640: 16, 960: 8}   # full precision needs about twice the memory; Ultralytics halves it again on "out of memory"
BN_BATCHES = 64         # training batches used to re-estimate the normalisation statistics
BN_WORKERS = 2
SEED = 26
SAME_CAMERA_BITS = 72   # fingerprint distance (of 256) at or under which a frame counts as a test camera's
MAX_SIDE = 1280         # training copies are stored at most this wide (labels are resolution-free)
WORK = "/content/work"  # fast local disk
DRIVE_DIR = "/content/drive/MyDrive/itms_uvh26_finetune"
RUN_TAG = "ft6fp32"     # version 1's runs were "ft6"; they are left alone
ULTRALYTICS_VERSION = "8.4.163"
IISC_WEIGHTS = "weights/YOLOv11-S/UVH-26-MV-YOLOv11-S.pt"
IISC_SHA256 = "12dbeb8f6fa207d19446fa652fc4313359fbec267027c8c01641004dcf1d0150"
IISC_BYTES = 57142241
SIX = ["car", "auto_rickshaw", "motorcycle", "bus", "truck", "bicycle"]
ZIP_NAME = "itms_uvh26_ft6v2_results.zip"

In [ ]:
# ---- Setup and checks ----
import os, sys, json, time, shutil, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}",
                "onnx", "onnxruntime", "onnxslim"], check=True)
import torch
assert torch.cuda.is_available(), ("No GPU. Menu: Runtime -> Change runtime type -> T4 GPU -> Save, "
                                   "then Runtime -> Run all again.")
print("GPU:", torch.cuda.get_device_name(0))
free_gb = shutil.disk_usage("/content").free / 1e9
assert free_gb > 30, f"Only {free_gb:.0f} GB free on this runtime; about 30 GB is needed."
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = DRIVE_DIR
except Exception as e:  # noqa: BLE001 -- runs without Drive, just without resume
    print("Google Drive not mounted:", e, "\nSaving to /content only: a disconnect will lose progress.")
    OUT = "/content/itms_uvh26_finetune"
os.makedirs(OUT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
HF_TOKEN = None
try:  # optional: a free Hugging Face token in Colab's Secrets (key icon) as HF_TOKEN raises rate limits
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:  # noqa: BLE001
    pass
print("outputs ->", OUT, "| Hugging Face token:", "yes" if HF_TOKEN else "no (anonymous is fine)")

In [ ]:
# ---- The 250 test frames: names, content checksums and 256-bit image fingerprints ----
TEST = json.loads(r'''{"dhash16":{"101184.png":"df2ce20cf8a9790c208c546c391c330e33229a30abc8ecc46ca1e480d760d358","1100.png":"3a15d999d8a5d3997e19c8e00f293bebef8bff21fc25f8a6fca4e9c0e950f110","110224.png":"fc6ccdca6fa68fbaadd8b35eb7aeb7cd3e4d8f5dea9dd5bcf63cc6f8dcd9f8d1","110866.png":"e1d6e512e312a2b286728cd6c4e7f8b4f89ee45e7264c5968b96e3986598c5a8","116501.png":"51fc95cd34f67dd47cf46fe4b344b3c0ebd066ca56821626e728ef88ed8c4ed2","11952.png":"29c789c74986c1cce3cfe74ce18c938c338bf38ddb8cf91df915f919f81df835","124521.png":"6224fa2426b98a00cc126c42eca2e0842285b2843080308058842c80ac88dcc8","127160.png":"2bcc9fcc97e4b6e4baf056b4527049b056bc56d851c44387739d57e849c87c88","130141.png":"ff8b7601f681fac0cec3b9e0f9a0eea0cf80cb10db92fb80fb28d5c4f94ce988","130501.png":"e762e6c2ead0f145eecd4e9becd0edf4ee98e290e5d5e6f6cf966fb666264fe3","13194.png":"9b9a0bb84ab0c598968cae56ccbb4a744a744b62d96ad8b2fc39ae2cc995cb8f","136336.png":"d6c858c9796e6128e4ecb2677026459a5e1f680ef20b220b560f5596b2caa9da","136828.png":"b88cbb0c7830bf30e838ec18cc269c0f960cbe0e898d9d86c74fe3e0c1c0d904","136930.png":"d226959e798fbb87bf8ef6cceecceaccd8ccdcccbcdcbcdd3fd47a70fa72da62","138877.png":"dcbedc9ee4be54ad9a2590d83492221233123aa29a82d2c07642e632e062b971","139810.png":"79e5c8e268f9f46c7036f41ff0c7e225e131f191f586f486f4c6f4cbf8a9f830","141456.png":"e484e584e106d404c42ae6114531a716b392b5929a92d882fd2ae524e3a6cd2a","142732.png":"0bb31e134d1311d292aac6e263e2a8e3ac66c86aba2aea0a5a0279603b207b00","142934.png":"c600c4108c105a20d80098010a01aac39b809bcea784b299b0c930e7783e7a15","144217.png":"e984a9a8c991c918e2529c0999219c429c401c0ccc2d5d1dd614b658f251fb11","149105.png":"485a656ac1444bba50d27772a69296d29272be421b6216c66dc06d404d410d41","153658.png":"b0561a569906bd82be4a3e486f212f305b986b8c2b8e6fc46da5ece2edf26c70","155069.png":"f507ed47fd21fda1cfa1cb91cb9acb8bdb89db0dbb44ac86b082aa2cc60cdd0c","158321.png":"4e744c134c921c8952c5596c4b0d5b449946b123e03bc932ab44d440d4c8d2a4","158964.png":"ccb4cc94ac1fee13ee33d215e24d4699c681888384807ed07e40fc60fdc2fd84","159347.png":"fffedffef83efefefeff62ffc2fb517f567d99bff986c1debdfef9fff3feffce","159579.png":"71287528b6687358635acd2b9d2b192b3b863ac2de489ac0bec0fe60b580fdd0","161965.png":"22e13709e888ce283c69f8c8e3e2c7628e46dec6b6a253d0edc0cd80fd81eb92","164069.png":"e24c69546614634461d0506870317058702c7016e00bd004d004ca20ee00ed08","165222.png":"eb6cea24aea4d89c64c8689cdcf4d8d0c8c27cb17091b1d1e5d1bdd09448952c","171003.png":"bb2dbae136a29e02d683d689b61a76097411b801f000e400cb0cd588cb98d718","172180.png":"ce788bb0c371c7329944a3667f24df247f103f383f783f74fe64fec4fcccfc3c","173290.png":"3ce93c697d496dc94e85db815b89de8479c0f8cad8e8d960d960dd61dd42da71","174671.png":"848e85d80c6f135575261266134257895361332a72c8e2dae078c610c7014cb9","175587.png":"b492ae91af52ae3bfd2c9b97b96bb92559225f2a7d9ebef2bca3aeb13698b739","175644.png":"dc436c02ac8c346236233e99a332e3d485188ec83dc0efc0ff40fe40fe20de21","178676.png":"6e3ee0b298c7999338937a0bf811b018b0323468646c6c1b2e92aa8828484908","179501.png":"d764d360d7669362d260d7605568e4d6c89ef95ef82ecf2c6a826b1878f3f630","181392.png":"6a68284cea607380e922bc60ba18fe11bd98ae9e3b83372f97338657d456e04e","186193.png":"c712d933fb2bc98eacc3a6436b2923897edc60ce60ce58e218581c781edc9f85","187590.png":"c600c4108c32ae22d500c5144720273866703530a598e081e18170c370037a07","188930.png":"f9ffc3fff81feffee8ff42ffc25b455f002d429ce1ec819eecdee8efebbc6e5c","189268.png":"623722555167d23ad2109230a098e088b659d349c768c724a5f2a7b20a5196e9","190976.png":"b9b7b1b279d079cbe1d0fcd8cc6eed0fc60fcf029d81f9819cd0e7c0d8489a04","19657.png":"f8ea3b9abc8cb74acec31d82788233226641ee61d6609c40b840a44089c2d742","200537.png":"76556ea9eab47a3ab27df63ea61edc1ed90e992f9d2f19a65d071d171d13bed3","207589.png":"ba2a58a9386cb04890c4959673c77382d6004d005b80a18065d0e7c0eca0bdc0","207692.png":"14f892d815da26d9ce5e9e5d1d5c3d6c3a74f372e7e2e7eacde5cf65df6d9f41","210840.png":"ed26e12e145430721c2a3cab272b03a803a865a84ea04b900b3083500348134c","22297.png":"fd18fd98fb15eb07ef22ef08cf8adf86b2a6b2aad6acf6a4f798e70ce304ee14","228433.png":"b493f293ba91fc30be20be45be459e261e941f140f4a5b087f005f0afe1a7e80","230788.png":"d6d4c614c457ced4cb449654dec69cc5a9c12b813ed0b0d0bc88ce44cf86d716","234270.png":"6e3ae71297d69f833f137b477f44fca6b8323c9b7cd92cb02c98288838c8788c","235615.png":"c71989b8302c3a233b33791249145d382504618c918ab5863596d2d4da90fa92","235699.png":"de103250f280e380c82db3257e91fec8fc44db00db305b105dc9da715f125f92","236776.png":"8f368f168ec68fc79b4699c6a8c6ad0c25b44e85b486735264b1a6a4bc88b988","236977.png":"0010000020000800080010002800100005802980214010400a00040000100810","241777.png":"9b41344031b01334aa6cdee50cc07ae46c6422c43238b29a83a664a298744216","242542.png":"8a2ac91adb82db26cb078a41db61fa68e2644268bc6470b041a81bf0e7f09ff0","252924.png":"b449bb1556a196909ad841b14c61633278c2ba30be2cac41ae09c815c067d5cc","253166.png":"4dfa5f983cd872dd92d92692eca2e4f2ecd4f0a6d9aed9da9b5c9e949d309c28","25900.png":"78197c18fe12fe1af60a68086d00ee10ae01c74c2ec6af129630b37897cc9fec","262858.png":"62c6a28620276403042a20271227110d28c5048d01d506b305230c7118711925","272400.png":"b605ce05bc05bd0aa80aac84ad9175917511d4b0ec349a5caa1c883ccd3ccc3e","274086.png":"e40248884aa08f09ab0535942c55db05fb0476046506b285cd56c4c6d4e6a4e6","276372.png":"d2c4d1c4d1a6d5fc4df6b1b2ec935d917d914d9ded9cf78cff48fe40fc50d750","278075.png":"71bc752c6e2cf3acee849b8c7a90d7406d81c9c29a81da819b88fd80fd40fdc0","283004.png":"2bf04be489e40be433e43ee47ae6c8e215c6f326b146a4669ac698c6d006f286","284326.png":"2d382e172e5a2e5a2f48394a29480caa2eb2425a526ab2733a773976916a9d6c","28476.png":"2604ac00d118e1088808940e206448a256205600a6d1aef0caf440e4c610b292","294584.png":"6eb56b317974704ce09deaa6ee06ce1f890f892f992d99a79d86d517dd93bc93","295604.png":"96e170e174e86dc24d927d98cbda8d03eb95c912ab563cf23cb23a94b0908990","296389.png":"d9b5dee5dea5f4b575b19e99b9b079e0b8f0b9f03c706e646c6a3d6e3674c66c","298124.png":"e551a952ad9495b2919106743a342318299c2074cd66910e92869f06d474d5b2","308639.png":"48b76a9329166d0368856414ec86d4036c0175516569ca6cc8e4b8669a64d874","3120.png":"493341a11600e548c7481040822182a127604f490f450f6fc76ce56ce6ede46f","314181.png":"365975d865ea777e5d15588fc2e56970a130a138b48cb45eb8c7e8a74e497689","326655.png":"f8248228f804fff8c01fd208db0c5905f23468c2e48ffc3dbac8f618744de0cd","326864.png":"bceaaaef714359f9d03ed99bdb689ab8b81cf026702370112004240030209430","329152.png":"8d518f9b724dd644c642c649d6c1cf608f748c741c721a601a687a6968eb78d3","332518.png":"dc12d51791959852de2658abf118a4cc62c6c203e213e25ae24ce218e308e30c","333449.png":"0c67259428d42498288a292208826823a8a054418451a0a1e4e031b051305022","337153.png":"e8b6eb32e316e7366274637167625771d783dfa35f6b5e6ade71dcc1f985f144","354370.png":"fff6ffe2fbe0f5c0ed81ddd09d309c905990fb82d301db28de28dc285b189b1c","358588.png":"dd4150e046d846d87ee6f690e5a1dda131aad325d23dd434f408dd48c5a9ddd5","364539.png":"a864da2e4cabcb11f611949180d066d165d92d9819991b99bb1879307922fb08","372670.png":"a7989bca4f964f0667c77e737ed138d1a8918bb999b81fb9bfd99f893fa567e9","375441.png":"c8144800482048004ca136223e1a302af022e149d541f100e1a0e0623002f002","379361.png":"2f495f19b75535517d79b5adb24ce66cd782c7238f254fa9d7b08ea06ea07f88","379526.png":"e982ec4366836711d661b026606542519131906c30182450e250caca8c910cb1","380524.png":"4a69265a7524ed236c33f71aa683ec693dbef1e8d38db38eabbea39b3b499af1","381330.png":"d4249624be2c998c1ca55ce48cb4cc8e5d8255e1bb45d6a6db53b32bb101bb09","384976.png":"a66ee4dee1d2d1d5979e1736073146f906fc8af02ee827d86d8c7a387624fa60","385489.png":"ebb0e27bebd6d95cab79b9f139703d6065736993f98af9a8dcc8b19c93388b3a","387455.png":"cb619a60916025a549a4ce24cf138f3196b99695ae12a733a5a995a5b8b4d9b6","39125.png":"2c043df45cc58c639c769df6c96e919d97cdd2240e7b6a6a426c9170cb93c88b","393838.png":"54df5cab58ac5a4adda7ccb36b39859ccc98dcd2c469e4bb9158e96d6c8cc80d","394924.png":"5946334cbad9ebda52d8fe987198f3b993f0b6f0b6e0aae8b6f0bed0bec0bec0","398959.png":"d6438ec6d8cbf18bec4bed269b2bd12cd5a2d5bac5eef310f2c0e9c0efc0ffe0","402314.png":"9b1053b65ea2abb31bb4c700c710c74086400e40ce758a7caa24a8a86820ca20","402694.png":"d58dd59cd394c799c28b648d4b6aab6ab9a37c66e4849d54f554b2f4aa8caa84","404094.png":"cbe0cdf0db52ae71acc87ba1f9a4cf25cb632fb15f11bd38b470a474c9e149a1","407451.png":"dbfbffebd7d98f5ae6986de3c5fe87fe0f341ffc1ffc0fec6fee2fe777ebf7d1","42028.png":"f3923352f3954716cf13ce1cb41e780ef416d8279a277a2f123f923482370217","423075.png":"dfa27e92ee4aac4cf5c0f5e9ee67ed66df68fc32f9b2fa2af9a0f930f910f994","425771.png":"caab4dc9eb27eba6e5c8e1c8ed88c788f328bfc09fc0ff80df80dba0efe0ebc0","430269.png":"572f153b9517b6c7b287ab85899499c699a6998a998a9988b9c8bbd89bdc1bf2","432315.png":"d8cf497bed8b4e8b3e3aac598ccdd9c53285e689c799cd89cd899bc9bac9bac3","432658.png":"9c8c9cd93aec339a7c0e68904b1c4e3a1d26fab2fa20d3a6d1b6d3b2f1b4f5a0","438346.png":"0b1242b074f2f664bd10b510eb14d9003882f225e494c316c67406744471cce4","438435.png":"e698c6a0e1c8c9c8b7087701dc839d0399039a02d202dc02dc02d936e80eea0e","44015.png":"07e893f833e02b30cf7899f8d370d370cce0fdc4fdc2f1c2e9c64bc65fc4df8c","451021.png":"d6c7f187ac43b927f332f3d1f148e1ebf020f938f830d808fa04fe80fec0f700","453106.png":"4f4aace0ae68a214bb09f9c46c4066626e38db389b16f38af289c6c9eec3ec54","46025.png":"0e569c328d6331c7f347736f724a715963586698c71941b80db204b10cf05c61","466644.png":"099063b25571434063407394b354e354ee52ec64ec64d866f0f070a4e126c167","466830.png":"c20ec2a5e6e4e4ace48cc62cee2ec62ae219e218e218c608ca89cb09ceacdee2","467170.png":"5408521252966388690d430dca417a41a465cd21cc31d821fa10f298b108fe18","483736.png":"df4549e3cf318d31ab114b31c3308698c4980c882c2c340c5484961416163246","485078.png":"cd350e711cdc35e4e364c7524f481a283b99b88db0c3a45366436461f520dd88","485297.png":"e5f0ca7e92a4d1d0dba8df04e39ff14bfd23fdb6f8a4ea94f2c8fb52fb52e970","490353.png":"462bd73116c834d477457349c341d4daecc664384c38cc38981c9b8099485186","505811.png":"9eec9eac9f7c1f3c543868c08bbcce780cf8f9f8c9f8cbf2dfe69fe4ffe4ffc4","506389.png":"429892d866b976b9056964d9acdcf94cf14cd26cfb6eed6ee326bd28a928f52c","506423.png":"5cf8d708b5d0b9d0d950c5408d0af102ff00fe80fe00ff00ba40e680ece0d4e0","508636.png":"a60db60dae0dfa0eb08ca3b672e43dac792caa2cd4b87978d238c744cff0ff70","514073.png":"9a65f8e1c3e147634d794d744a746556ebd173d2f3d083d243d2c5f1c6f1efb0","514122.png":"d5a5d6e5d7e1f4f1dc55b685ec90c894d8d49cc6ecc2e8c0a83cc880e800ad04","521626.png":"648728854cc9ca2deab56e95ac635d035d69fc38dc787278b27c64f8d4c8d468","523758.png":"cd662d67898d8116c65ce13c61446b64fb15c13d9c7876d2cdb6c9b641e4536c","526014.png":"d249d3c9d6107750b1829261dc20c580cf80e780df90ff50fd10e940fa88f288","533942.png":"c119d919e599d4ddd45c575463102381a32393a193b493b08b908859cc68e628","536839.png":"ebb47fa4b3e4f2e8fae6fb949dbaed1ccd70cd64fd90ec60d8c4d100c6149090","541303.png":"96f3a5efb71aa50ba711d325cecda310a349b602d290dba2cf262e3416240e6c","54303.png":"63424b5266c6524148c05310f58ce5e1e1f0c1bcc1b6899bc454c12ae111e188","544842.png":"d31fd70ced06d625b7206e14df4abf1abe06ff43fe20f800f110f402f640e548","546564.png":"9210b134b3669742e6349c7e5974616ccad88dd23bb0f760ec60d6a8ff40dac0","546774.png":"3d6825685d490cf56decbd84f325c3290c65186de2ecc4ed8bab0b6a4a7aceda","547253.png":"7192e89a0693ce9135b1b9b0b3dab7f8c778176877ecefdcdfd4bf94df248fa6","551599.png":"7d867586b587e526c826cc64a6d256229c8bec83b852ee3431517632b5d54d93","574487.png":"167084521650128c9e042956bb201b34a6386c184e0415071c892a1b2e126634","581339.png":"f89cac99e71b459bcdb3cc53e48acc8ecb06cc4ed68af4d6fc16fbc6f356b746","581773.png":"5e297971e191c32a855bcc64c9548544a50532067a0f560e060604029403c923","587187.png":"d11bd98ad2cc3e46352139291a741a54193c1929192d5da45996b10aa303a301","589684.png":"dbb9ffebdfd90e59e6984fb3449c9cfe0ff41ffe0ff85fec6fef2ff767fbf7f1","606053.png":"2fed59ee3ce69d6bdaea9aeca7cadfe6ffc3ffc5ffe6e5c6abe4ebe4eae4ebe4","61403.png":"e504e560f680f9cac4baf630dac68cc8909880b08092c0b1cd91c599e599e11a","629029.png":"ff97f53def355a895c153a6ab93afc4bf853bc23bc25fe18fc18ed18dd067d06","637122.png":"c8bae59695369534963418342835d0edb8c4b9cafa80e4d8e5c4f1b4f394d3c5","64003.png":"6cfc6ad8691aef0caf96773bb72bf393c3126d135b19d51ad51cab19c208e320","643563.png":"5aad199c798ee3cea2c6e5eaa5d6e993c5a28598c5d9c558429ccc944d992d19","644081.png":"4a4f52ce4e8e4b03803bce225024660861c6115890dc9290b481ae4462a46a02","645326.png":"b50475467d437903f981fd90eb105b88f388f78cf708f710ef14af14cd94df94","651449.png":"0ad84510833ddce3f1c825eb2f6ab7f62fe63fd257d33fd33d983988bb9ccb0d","652367.png":"1aac9e28de2bfd63f86af923f5927f93ee28fc00fe00fc00fc00fe00fe00fe00","655387.png":"dbdadcbb3e9c3a1eb89f7e8fcf46cec86cb476d3d3d2dba2f1f2b071b069bce8","661657.png":"7ecdc4d52d4e696cb95c69ccc5cc9a522e127e92dd97eb92f5b2f38aeb32d72b","66500.png":"9e10ba8aba88a998ad94fcd478da784c7c6afc619a61df44fc10dc00de485e60","66832.png":"d952f95552425b69992b9b7bad29a9a06db81da41ce51c6d5e79d0d5c8faf7a1","668419.png":"ed2be10fe217eb0df21fe3979b9ddf0cd70cb708ac261e12c604ce0468003800","671140.png":"b278e6606140e440e6a8cb2a8b0a6fcac706c0067c825cb01c0459005d007d00","682810.png":"e803d0b3728a642164269900b98cb98cb38ce384c384cf8486e6baa48ce4ed20","68330.png":"b16c04be98b6837688358035b21ea018b01c7ade78567c17fc47ff277ea3ff22","683595.png":"452c448c468d952633b4c4c6946304319e98ae1e649e464f4ce348e5dcf2d0f0","686114.png":"80ce346a6c1a160634d23652944ea899238343826583618b658b368a32c252c2","68903.png":"99979bd7a0d5b2d9ba58aa668a4aaa72a852a473a41e921e9096da8e5c18e360","692196.png":"1b4b530a4733d732a432cc239837759761efcbe69bb3b3a94f64a644e724ce40","692303.png":"c63eef56d29cd998f999cdcc47764cb06498e318850c0cc709c733e727e14fe1","69976.png":"726472e8b8e068e070466602f802a002c901940cc100d90cf994f3d0f400f010","70239.png":"516813ac4eaedd6bd5c8d948dbf8b2ca66cac9eed145a1e18d91cba3c7a281ea","706259.png":"f0b6259e228fa50da11a20cb3464926a918e6b5526941abc8938a73027a0b3e0","706931.png":"e65899d8da9ac95a3ed1ddd45c94dd18adccade4ccf9cef98aee8af59e536e5b","712192.png":"f66cb65ce555d763da0dc288ced8cb3664764c324c326c3c2c7b1cf94c790d78","713343.png":"6c3ee1b299c7991338137a0be80df804b406340274006c022e10aa8828c84a48","715242.png":"96983614d22a950aaecb2f086f2dec6daecd73c933d97798765e7e5eff5effda","71748.png":"ce189e5b3e0dff1bce093e193c1d9e1e7e0cf84ce64cfa08f8e0b86438a43820","740400.png":"33a1768925a9ce4abce2f8b0e3eac5a6e5e6f646e6c2c9d0ddc0db80db11d913","750600.png":"de707aacb3f2a282b1ca7b027682cc26288cad2c75226a3668b6592279a4f628","754472.png":"7b246376431763317160cd0a5e467b2466ccbe70bcc4bca98f59d5ced7cec717","756528.png":"ece566c0ae82b602b0a394aa5cca3d6c9c2fbd253d110d852c86ac96c69693b4","757654.png":"710c3184f4846c84f84475c554a0d4b1961bba1bc3e3ee43aa42eb06ea06eb06","761669.png":"9254bba5f539eb65ca8a7b6fd46c846e94eec5eec7ba877437e874c8f9a4eba8","76552.png":"c1f2e1b2f9ecda794a75eb45736bf2f9eeb2cc30c838c976c826c4a6d283e908","76645.png":"12349235a55438784a6451ccd90d20cf7e5881411519a59c5dbc34fc6df407cc","773162.png":"c598c894ca32cab4c8a5cb145c10a4507498584859945b064e589858287039c1","773806.png":"9d464dcf6c4f0c528cde3c675d15bc25b926f44e74cc94ccf41c549c5c965cb6","774596.png":"fb859565bc455969d069dc6bcc4ba72b908bd162353d1519a31a83358b1d4b7d","781746.png":"eb06a180d999e3a4c6d6c693c6a7868502a312ab14a509a51da02daa2da23da0","785007.png":"1d58598a4dc87f2c6fa47de665e672f333eb4b6c7874b364b7b6cdd9eeda956c","785732.png":"e308eb8972a033a47391e131eb4cec9ece1ececb909da6f866592b390ac1138b","789065.png":"3aaf3b473ce13534b39a32cc78e93965d971cbb1599851d94ec8fcccbdc17d88","794461.png":"47747b701eb4d4991944736675e2f5a2d661c663a661c6738f626cf02b60ac64","795287.png":"58db6ab9ab39b819b458b8487a4e734c7b287b96ea70d360eb11db018b20cf80","80005.png":"dcbcdcd6e4f674a5d86599f0acd22a522352658260d4e2c868e47cac2532a671","802597.png":"b3a63389a3ccc3f0c3f889bc89bc8b9e139f111d990d998ef9a6f988f5b2fa90","810331.png":"f82cc7c8f806eff8e80ff21eda0cde45d51ceb64fb64e3a084b49c0aaa9caa88","818164.png":"ee3ad62af5b8dd3dcd136d73ed51eddf58dc5b1e5b4b330d9699c318c93af51c","818557.png":"c2b0fec07e48ef22e371d3502b54b690b39013a01f023f20f3a4ed219c22e3a2","824427.png":"f349f249ef1bcc3dcca9d9a9f129f151e151c149c36523512d109f40bcf07850","827304.png":"e0ccf038f650d748d3b87778a879c138d0fa487f081f9a1f1827310592081180","829240.png":"dab6ceb6e8b862dc639475dce5dca78adf4d950db70436847f4e4a8ece0ccf0a","829959.png":"ca2ade12d3a287047b623a52aa226f13ceaecb96a64414684c74316630b1ac78","829978.png":"79d16c946d9c3118233ab9ae3ce658d4b974bb6cef0ccf2cc3b8ca1ac61accf2","83149.png":"d6e48e949ad5bf31d9d9cb99da3431b5b1b7b4a32c496029d261801be01aea38","84796.png":"df3bdb8ccf0afba1eec85eca5fcadfc4fee5bd63377abf309d549980b130e128","848486.png":"f3357354f25cf294f6506c70e6a0ce22dab2e90ca92cfb04ca04ea16e85a991b","85483.png":"71b87528572c1f281c2276b0f6b2e6baf5abe4ebe08bea6be06ba169e271f1c6","858005.png":"81f692ce72de72dbd356946355c727a72f6646fe41fee3fc4c3c4818fa39f879","858355.png":"1731057647d217c6d6504198f894f53ef627f869bae3ed60fda4d728e739cfba","859158.png":"5a6669119023995727c31fc9d34d4501ed895d955a61dc39bd637f52fd52f09e","861398.png":"60ed602d206c580b282f4812ac1e240e861284c38ed38191931d91ddc8bcca58","86218.png":"b58c82acd24e58f2f340e301ea01e004e004d00c8218c830e270c4e0d3e0f7e4","864056.png":"719a58e2350c350972e36279532ce10a990fc90e90e7b461a421e43ae45ad633","86480.png":"895ee19e313a33337f176db56fb9cad19a69dcada0cca453d4646c057610cb3b","866078.png":"5dc7f5a7d6c3f9d6f34af3c1f152e591f390f918fc10d890fe00fd60fbc0ff40","875190.png":"95359575d4537c38ba1eb9257b0d7b4d7b2dbb2d3b3d791d5917714623626370","881888.png":"2c6834ac3424383422b26c664933d8c4c0e748110c19440d441f441de43dd23f","893031.png":"471d471a5b1a7a4a66a3ef8083082e00bbe093cc04080920923084108250c450","89437.png":"a7a1be9df28686341c44f9ace126c7668ed69ea2f4d2f3c0eb82c583f792ab5a","911150.png":"0f309e8186e80e688e784a4c5e696b5936182709430a5f065d06ce00ee806420","912687.png":"7d8e75863586e526c926cc64a66256021e03f843b102b186d681c52a166b5e22","913429.png":"c854cb498b4c3bc73bc63ec6e8c170d0f1cce104c1649326d3a3e185eb11f111","91587.png":"2538e4104370cb70393419761b464bc88262aac2acd3b0a2e422e120c124c272","916637.png":"c608c4108c141a28d900b904bdc0bcc09ce0c222c222d2d8d270523c5e0e5a17","919010.png":"3a30395431dc7309b879bc31dc69d9797993b9e3e3f1f7e3e7c3ffc37f027f02","929272.png":"97d1973986e107a027227702da80e6e0f2c4fac4fcc8c19ac7d0e7b4bfadcf68","929878.png":"35d0369e6d936dd8f91aed2eedb1cb2ece13cf28c76a464c8e04cf844f441bd4","929978.png":"195118d51967182719d31c721c131cacc93f6a7e333ab3ea87a0cf9cdb0c5b36","931597.png":"9b768f968ed6afd68b5693d690c6b3cf139ed5eb97b78ffda5e4e5acf37cf239","933281.png":"bb2dbae5b2209f32db8bd489370876192c19b919f208e4c8cd88d988d398d718","934870.png":"8b619dc1cff1bea8ea68366836689ae86cb2cc9658381a081f489cc85f086fc8","935810.png":"cce0aca4ad66af4caf08c091a999589ab0d95899d35a4bac0386c682ce0dc72a","938125.png":"8f768f168ed687c6875615d60cd6434a58ba5b9554463ca2789172847224f369","938473.png":"712865286f48fb28a59a9dd23383ebc16fc06bca6bcace9aded89c8cbde6fdd0","941351.png":"68686861e96ba2c3a518484cc2c53287430d6343420a73818290ca10504114a1","942845.png":"ffe0ffe8ffec73c47fc47fe843acdfac7dc5fbe1fe60ee63f463dc63dae5dae4","94361.png":"d135dadd5a5b3ad9ba757e71f031e130d071e4e1c6f2fbe4b5e557d05fb057a0","944502.png":"792877287368f728cc8a2f892f9137d137d9cbc9bfc3f6a1dec8bd80bdc0fcd0","945271.png":"722006604e68996e352ca926d9cb91c836486f2d4b943370d238f018f348fbae","948729.png":"370e67489a829d109d181d0415041582598459a569a0e9a0d9a1c99059d058d0","962847.png":"23b769f6f9fbd0fbd6d8b27ab47e35d72547c5a760f23d71339130f730f6b474","964462.png":"d30aca2365b25d45764c76ac2966bd76ad6734a733893e491a029a4a9a418a41","966551.png":"96b3b577b795a58187e146f1432a3de72b9152730e3926183e2c1e368f378f77","966699.png":"6ce217982d516670367295631655c4c145845b112d899b648a55bac96cc86de0","96883.png":"ea8afb3aad5a2d6a18e818f08af09ef29efb97ed57eddeef9dd79ff32c671b36","972582.png":"d59ad278953849f03bf447d4f56c8d7caba4257656ccdb04d0e6e3c6cd665acc","982198.png":"c948ca4cd5549647b1307664f09cb64fa4c7c86ccb18cd0ce176f332f2bcf138","98292.png":"dcb6d5c3eba1fe49d72ab553ab526833be2276626864ccc0d8c0d390635027a0","99206.png":"fd98fb88f5d8fd45eb72efbace6ade67be65ba6ade65d87cd86cd8acde24de44","997377.png":"cca80a8a1e0ab80ce082e8abc9629940136009784cd00cb086108911a09a6cca"},"sha256":{"101184.png":"77d47f52c15811c69f77a4544fa409c7cbc60869465a8857c1390b01005d2e94","1100.png":"fac27debf215b5d262f096c9d098f68b557ddc8297b60470e283f1874d05fc0d","110224.png":"f92999a0531708b7a6bf918265f13657c099af0808d34c3a6db6c0bdcd4da9c9","110866.png":"70ba7808e84f4937af103dae18368edcc2ba4ffaac61704e412374a61e054329","116501.png":"a9936dd14367689e2360bc1a3769a614c1fe4fa8773dee73283c857be505a9e6","11952.png":"eb9f01e0dd6f74a9181b3b29251bb0bced3bcd143832b75640d81003ef26455a","124521.png":"29f886c54fb11d8a8e612ffbf18bd493ff6c1b9812597eace12153d4a5e43a40","127160.png":"54b36051b5bc7bae3294d12c718eee0595608b8caaca8d2f9438f3fe72153308","130141.png":"c62991daac6b03f4d6948f187b43e4ceda5919c98268a20658f8479b12a03d94","130501.png":"52b1e226d7dca22560f8fe8be27ccd099693faea9d48b72ba7d2d52c27ba9177","13194.png":"c3e8f44b219ade4e5a813748c27107ad7cb295d8309e095c098bf7074a333849","136336.png":"cc0623e5ea15a61ed21ddecfd9951638f21906ded7d3694c99ac19b937d300c3","136828.png":"500c9b7d903c61c0d0a115ca7d0bbfea3ef9a205ed11b44f9af80e1ec639443c","136930.png":"d22cb424cb64b465b7866d406a08d88b460491fd1637c23f3cb982d9caaa8c1d","138877.png":"4c47915b286296262f59adbd1eece1401ee8801d83e7cd47b875d0772028552c","139810.png":"c56e382ed16b749d38b2bc077500f6dc2ba723c692738162b028529654382916","141456.png":"e22d7b7115452e6b5f800c928cd1d5090ee23c0f14004bc432db33075a39f16e","142732.png":"4eaebf8010deb4b4aced3116219d5096513870fad4b2c366d6dd54e3c27fc64b","142934.png":"0d43b9b9b4ba520c079aa0f980a7cb838168083240761550c8a29659f72a4ba9","144217.png":"2c49ca9e8073dffd0b9335e4c1f798b69f673f088f4a2d6da93f6de7a769b0a4","149105.png":"36572a7ddbbdac275f9e0e0f1acb9358288d1b4a42fc3209aeac25d68266ca0d","153658.png":"4f0000f25faa4b10540b9c829482dadfd276c47e68a05cf9e89c48edf8ec289b","155069.png":"19f1c99b291a80e6fe6d9cbd2a4f4e45cefeaf8d2d10cf6b14c8abe93856b90a","158321.png":"5a67565ad4b68dd951a4884b9110b8fac6f272d97b2b0df98bea8221a55e61d0","158964.png":"9b6caf9ef87278f447eec52dd969be257755bc6ed8434ac738ea8b8cdf89dc8d","159347.png":"23c24ee3a61cad869e61659cc5e95a682486be3e8e70c36bfe8edbeb7291cc7a","159579.png":"f3c31f57520107b6cc428c89f3cdba8e88c451e0f7a8f9a65565e5aa724c51f5","161965.png":"490267c34f6d09bbdfa9799e2674e61a4a6b3c27d7bee5aba775fe3383312dec","164069.png":"37433ad0e39cfae7a272eafc7fc02549dc5990122729caa36d77e08edae9f091","165222.png":"70785e8242d4554937d5908df7d0936512ebb9afcd8bb964f11205b8bdc4d1c9","171003.png":"7113454af595c752fbe652aad283e6a2bf0810390367b047e08afcaa7a405656","172180.png":"a4f3b991260c1b00db6c42b7d6587d998cfa3acf06415e48c12efad7ab776828","173290.png":"b8ca0cf05a7ff588f90c4560bcca1714eb99f50c4bbb2d69c1ff24dfa81d19f0","174671.png":"c486a52f8dc39d07f5bcd4851335750d13af6fbffdc953fa94f56fae38eea30b","175587.png":"1f1ca4dbcc83ae60e35de5f2528c7d4d827ddd3c39d591f6fad53acda9aa8a11","175644.png":"a2aa886b5ccbdff7f33fcb3c9429e9f02c018b1fbf19e81f2dff41bfe0b77ac4","178676.png":"1188def8433714ce7db0643d911d9f1de3e61edc562403efdb9105ffc9e3ff08","179501.png":"95d3a1cfb30e2943c3b0b13383cd9351c6b444671edb21d55abc951413f15e58","181392.png":"ae9da9f13c93ea708bfdeabca1ee6feceb309d3b9480d0ba2d94f7dbd54892a9","186193.png":"e48d419663eb83c4c15cd61f0dcdbb1fd6c451b5b062dbf10d3074dc59bc0f11","187590.png":"16868212ab77b1e2477da9f8aaf9844c2fac2776ec2389adc9fbfa3eee3b7d89","188930.png":"258cf534f8be0212260f8880c03e5acd32bf97b11d106daa87388fc00753e438","189268.png":"f77aab386a73836091dddcfee808bc8e7d4cb884b9c417447714949c87aca058","190976.png":"146ea316846891343e2bee55b53f472e8e8422cf9a3de65e73c181b798afbd17","19657.png":"5844e5e9753271ff1ebb1db9bd39c2c58498ed418c16f88aca933019ca5444b7","200537.png":"7a7aaf777ca7d3795b493df1cc51751ef922b2cdaf495c93b3c79056122b5ae8","207589.png":"826c5cdabbafa557eb51d4870f99da76fd6cce31d7e9e0b907aa5fe2bcc49186","207692.png":"0bcf4e17d6d9e0549dd6f087cf154fb3f6c9a19db484121056314d46bd5c7382","210840.png":"cc3180e2bb28268376dc4bf1ec137353c6c3385891a9c0ea20e2e57e7a3ab6fd","22297.png":"3789e904a0bf040cdeab168014f02cdd04510b379656f36b0e70c444d4204b1e","228433.png":"a10ef320ea002f17491469a64e96e7575ff2679f326aebf17bb681245f4782d9","230788.png":"d7895fbcaa4d88fcfcb35cc0a379ab001dd1600ae79c3cdec6a481380b0d4869","234270.png":"da679f70e532bc078614a3b815e681f9facfdfd16d1e04e249b3ae59bced928d","235615.png":"1443dd2581f7bd1a766c2d0dfa713201b99a278f493f663ff0aff7757c3e70b1","235699.png":"7e8dabe3e8a2b50911633f5973e8d0be0474d5d76abff6d40d26ca525dbf072f","236776.png":"6b46dd0103a95ab8d374f53d9a790a1f3bc3f70444f29175ca80c1e4eb5d7ccb","236977.png":"a20677e2e8f42d3c13bec8672c2a3831a5126e626ee22eb6740753405ed57d12","241777.png":"959fb0b8591ec72e71201de6ec9f960567e11db2f61fc5e96c5e31a9cf4afe7d","242542.png":"8700c256960b12c3e404a92d23089f75fdecb0caa917e180d357e3f524254c40","252924.png":"221dbfd516eecc2b2d6534c9a88b11c6249bf8a9e257ae1b62f9fe74ccb2e72d","253166.png":"6124b87540049f3cfb044526f3eba3856caa312ca58fb59e97104882c3b3c9b8","25900.png":"e192a4d6ded86cc32c34b03242577678c3e4f67b3d73acaec9f79a48df5fa384","262858.png":"d70096788ab7bd12a98d2285b28c371751d12825ad4ce55de91ee0700da46b10","272400.png":"3c0739fe7a96bef1dcf89a417d3502a08033f226a4a81c48c34f78094e2182f4","274086.png":"5c0dfd8baa65611ff5f4b0e606bf4d2c5e1e3bfd490c988024b79db38f872f2d","276372.png":"ae5e38e6ca7b579d91b9cb0fe9396f528444592fa4084249163ab01e9f5833ef","278075.png":"a0f7a5abe5ab01e2092163ebb084c35e523a5a5dc7689a245f6536fb85e138bb","283004.png":"5cd77d5921884d8753623e204b33df4b91da6496c2147354f7cdbe6141e45fbf","284326.png":"446989e39fbbb704b0a3dd89c705ac982fcc6831320d21f1219e9c1281592edc","28476.png":"b89a4ac0853dea03e63eec150a2f4b07e914d63a30b75785913f192d586318a7","294584.png":"2697f5c2cca90f84a6ecbd08675d12fae326dc4c06b1a6af6a002fce620e4199","295604.png":"2973e9d133cf6551e9c9d92e7dc49011cbe0190dcdb81c2950890885f21a9ed1","296389.png":"63444bf736e5f80a5922231b6a7da46bc1c796c3172bf3d51ad4b5714528d975","298124.png":"c80870eb7c7aedcc729144443c1df5de1ecd731a3bb7768326824fdfd39eb561","308639.png":"0c40f1a4c88e609629c0ffe0f8cf06ef05e88bff693591df0ca279459577e3f1","3120.png":"0f9aec468a3a4b15efa9de91b85c8a26a4ef96f4f5de44e87f0830e42a31c315","314181.png":"42154477508337cb2dd4ccb97a3f9eac98781c2096b7809ca9a81ebb1fe36854","326655.png":"ae32c8c79af8313e5531f39e7fa1e484571320c10c97f2e5d0e7be9c72307476","326864.png":"5d66cf1e6bc2a570fd23e7cffed2b4e9ac8e50a43fd0036a5790ba6de6e440fc","329152.png":"34634556d04cc792881161bc07ad4d19cce15612d3e02d860f4dade5c45b2026","332518.png":"ee54f4b10dc5f3283caa1fe89647f4275c3c3e1c2bce4c8d6ef2719fa5d4945d","333449.png":"1c419e07b37716d23cf32f67724dc5c8cde3020225e096513922709993740211","337153.png":"5b91f39ff4751605ba9a65342ae617a178cd4aa153f935b105fdf0a5a14fd30d","354370.png":"d2c2407747e0b5210369184008db1bba758225724c7c9cda5e29759f72b912c3","358588.png":"c055dd590579a862aef199de555bf69069f984bd7e44b7a9311c6396bd6dbf9e","364539.png":"838ef4b09cadf7f83f3a1bb8495adb98a9427d31d576c44f33b7dd7b3ffa8a60","372670.png":"c3748540348faf017cadcd2c7a10811eb5021ea1bfaae7fd6c7f81a34f842aee","375441.png":"d3bbfd847aca0ad537680e05642c615aa55431fa55756ae269b27882a5c64c94","379361.png":"2e3abfbea8c239e737cb0b4f63a29eef2eedd897bc5685f962422ba90808a267","379526.png":"41023aae0391d0e19b875a2c99357f792c7cf2c68aa6554c863a4a9b0861ef25","380524.png":"bb9cb81f0df4e696986ddec4a6d270d654a1ff4270f24acf51d40b54925870e9","381330.png":"67b3abe45e5293235f303092a36800c6ed81e041ff877befbb6ba4d55057d449","384976.png":"1d1cb3f4a6bc2a68fa8c3f7cb5c0dc8d68e157bc813a4dec164293c5891b03d3","385489.png":"353db52ef1cf59fc30b9872bd0b17973e65725b92ba14f87d257c2334aee3307","387455.png":"d4369eb8a71f9e6652d62640bbacfefc7bd658b0314c62bd466f931eb96f6a23","39125.png":"c373c6b524f6abc56c504d922e6bbde8727db1873c2c19a4a5834a892d50e031","393838.png":"49b5a6a510466506a9525f41878515ad879dd5788916fef266b6a74a00f29c26","394924.png":"bc028f78ad273f0acb876b5468de2d24cc768283ee1aa1c2278de8f21b08b4fe","398959.png":"0a77fa5be07a8a9aa35d7d4c94af8132ea5c5b25d4cb125592df3904c3136941","402314.png":"8c983362a41b9837785292b6beb71fb27ef95336aed5641eb728f373c1353ce6","402694.png":"1057f38e05daca33c25d9aedd560af02a66c08479b3453cf0c0fad73f8f24161","404094.png":"183d78407c5cef6462805cdc6a915179395282a09c44b3f7a1a21b68ef698d52","407451.png":"b854efc0b3d323631f6075eec15d14e251002712701ee5566145ac45e10e181c","42028.png":"046d21d15c5f3b8a130112077f12635b65647beb0358122632401e9232d7447f","423075.png":"bc5d9301c039ab7e8acda6c6f72d5e0ecc1ec6625e3b7654e0d8134d945da40c","425771.png":"324d2b76e3af50c81a4097d7bec1c15d4a7c7bd99bd6acec0d2aa5840906dc65","430269.png":"fb9b61636dc18ec690854f0f9afb8a5dd0ba87d901dd4ba135b3fe05442fdb81","432315.png":"db3bc2926b8bd378d6024fdba7f15f82e28028524e7a32d621915111a5b7a282","432658.png":"24ba09f7d71d8b9bee7a14922b71a4968704cf7a3a7afaf8a017e67692222bd7","438346.png":"ae7dbbb985e78e573e75cb136557b038f7a6e855bc3b9f60edcd7f092b24e761","438435.png":"77ab25196cbc42ed54a3dd7ad13a5a2ce2ba1131874235d403f3947ce27ad7e2","44015.png":"308c66b290406c02fb55f7f3fea9a73a16c04024a984547ee26652087f915392","451021.png":"32ae6a9d5b9ce0133ccb4a69e57a69f8a6249469653cf8a06e2a9a6027c2a239","453106.png":"750044d3378dc2ee24846fab5c404dd4998440385dde03516af50d2c9082a559","46025.png":"bf8f6fad31a4a3b32d75928f49b1fa42667eec508563b3530639ea27e0d7fe74","466644.png":"45d0b60d2e47e4427d40b272e79d8e943450e7a806f0b55393c5fb9ad385cd4b","466830.png":"eba7711a56086ef66879ec1c3e0107b13368a85225f39b79abea1dbf88532c13","467170.png":"6a92d4906c87d0b7f7b03180c3dedd76e6cf36040aa51d3a776299c821e899e2","483736.png":"07107cb6c96b684f1639010cb3f12cb9e2e60562681bfdf9bcbdacbd1b1f40b0","485078.png":"62ebf9f74751076d7deb9818190b8b1245d5233d1e6e074bb06f8cd5143ecfb2","485297.png":"2018ca35fd55cacd2025828981d59fdd55a210d287be2b3ca4c46961358b8436","490353.png":"f1a00ec6b334c18ddf523eeab277f123025ea9372b225e90c4902e622c88d4b2","505811.png":"41b432f49eec57227c5f6e098014b6db5f86f70bb0215a86e982cf7146b44a85","506389.png":"dd23c6aa405c57d3f0d728a0ccdcdd82bf6b57f2fa4c3689ba4e02c8d77bc7e4","506423.png":"b59eff046fc06e4e14b4f7d950eec2ff5fb3e38beb6b51ec6f6c0496228eeadc","508636.png":"75bf19db355476b376ce272eeb047a7904e507fe526d72a7c6d5ff663d95d004","514073.png":"71b5953bac716cd789e83ab0c1b65dd18abba55e8b137eaa72df475ee8662666","514122.png":"dd8ebfa311d8f5ad6fc3c26e74c1f37ff1513f7b4783cb9c2c78363654cac3bc","521626.png":"66d646c53cf70bd364e1ab9d7fef9984d0a9c09b963a57fb4058e48294caa798","523758.png":"b274546c1b6ecf968eb00e86a4a07ef548589c06226e6f22c36951998917c027","526014.png":"6d887b373552a152e54d65a75ecdd0165c3364b9588aad6870f4ded1021cad7c","533942.png":"087bd887d7eb40cea6b3e4231ba71cc206e260a5550999a2e16c088d38d14de4","536839.png":"1054e7837297ce0e9dd08a96186ef2206ae441beaf587d285e7d9c709f72773e","541303.png":"e98f2a56a030f12cb844f47a2082412bef97d0f31abb693fdfbd00501facb20c","54303.png":"448908c9c7efd5f92e1e59c337a1cb6703f10a196f17813a6c2e10168d2fd34e","544842.png":"17fce385fcce3a0d985f2bc493729722f3a14764f3ad88b464a6f21f1e1c5fed","546564.png":"78f1b315b35f50a39753633df547ca8aa7e7f8c98edd01bee7c326219736e701","546774.png":"bb67a4f50c2b0b68de398a8b32a5bba385ee621d4e96d2faf93c9049a8299330","547253.png":"4cae467ab7e627d80a7dc93919ad752af964f6827e688ebc5df21c0d610d5114","551599.png":"7c192c30eb9f3b8789976ed082d9cf7c4dff9a4a5a4efc19fa413cb6904865bf","574487.png":"80d3b138d9d203b85c1708de987a6b6cf2265b521fd85a716cc105d3e261289e","581339.png":"70a78e0837049ca3bddef0d8dfc90abdc177c2c7a0140dfab93521f2e86cab4f","581773.png":"69395cb8c0840e575e1321d60aa1c9e8b4de7804900edda1b298d458133fc581","587187.png":"3998404f1ca68e7fd00977c8e8a5589745ce3183e451b5ceac2175583acc1bf8","589684.png":"b1bf88770fc5ded8e4fec66c0ce68ef167d8cf086fb8e6756d509a9621de88be","606053.png":"1a8c9da7b5f75b4ab30ae89c020746e3ee770d0a97d55e0209c0efd101255b79","61403.png":"0862822c62b3672c8b0779ba033fc4e19b8f4a747fb68ad5cc6032aece1487b2","629029.png":"0e3c6313af16ba5cb08ec5b4ea834a15f6040b3562ddf5d4d2d7bcdb6bb7ce8a","637122.png":"1e6ce4a75177ad436508815d723eaaeab252b1b4a47f7957dbae2c15e945932d","64003.png":"e2ab3f244b053115de9a4e84c6e56a19fd0c040a5a1938abafbc52cef39f71ec","643563.png":"05c4527ba47da0c2770ba06f1629f43e9f872535bc720e52f0f6f92854639e63","644081.png":"d8e3da5efbad1ca0cebdb12d3c74e0fa9fef95a050074817454e0f198e8c57f8","645326.png":"1537393441937ebcb64bb92289ce1fc92d362e6487b70478a2fa77141c5707a4","651449.png":"134074976770943b24ef4dd0ccb333e235fd5e631d352286c0d2839d8e45ad74","652367.png":"9d95742a641dedb0580b45a8551fecaab44a3a93e485ae748b67a9ffc5092cec","655387.png":"cd573996506e6ef77118e1fd3aba0b90577994b54231826eadedb2fe0918b7a4","661657.png":"391e53f78f60eacc536cc106789f494a08052f34c32f59fd1089c9530c7f0003","66500.png":"0a9095b682e244792de6a50cdf3d1461d4d97bf077a10024530d6af23af169ec","66832.png":"5f59f37df070f52d14b9429853863d3313da08d9d11f96dc1d5e977322936b3f","668419.png":"aaa9583412cfa9e1eeb615968acf7b6dd37ca3d1acdecb2deafb4ede5f984a61","671140.png":"1b2b18fb48c0098aee5eeed80df497a0fff558e88ef3864be9cb51f244eeb9bf","682810.png":"e97bda7854e5ec0511d32ab225f7df566e54587e3c07ea6b0de623e16f12ab4e","68330.png":"f8c1c70e7856bcb78f47ca999c3b72a34ef67d1c00bd8259c0051f7c35672ec9","683595.png":"37e6acd257ace915743b4deac3b3bc4bc85eee270d5db96caeec07a6f1ff8e3b","686114.png":"7a06a70a5593ffccdd0fa19ab19b02b3475fb0c46b2785d1e629b5c0b1b8b510","68903.png":"ad4c4b64bfdf524eb520ed7d89f57e68b01c3c8af4abe7ede6b720a16780e89f","692196.png":"98da63254cc7106f82a61952ab7a2f654d23bfeddff82aaf4220ff210466132d","692303.png":"56b10832c6a6f9abc55d2b957dc40b0194fee9bbc75754d6671127362dabe155","69976.png":"9f900dff959afe24a97e89b64c2a6cdccddb4f37678ad9ca32720814adf71cbb","70239.png":"2624f4e31172dc7d1fb2f792426fe2d1271af09dbfc689f9f315adf0be3761ac","706259.png":"7e9357b6f4148cf5c94a3d30a371aa0c2a77bb1fc6e7a5e17365bf80a530f660","706931.png":"a2075009cae2fa904fb8753b6488bad7b739ab69ca8deb4fe9b0995d3ab691b5","712192.png":"e1c8452398829ee2efaca2220fca68762010db45d6b5492fc03dea84e8a3b6f1","713343.png":"fa48176851d25cac08ad2f2964c8355f2f05402583543c4af8a54410c2a24356","715242.png":"58877435a8e04893694fc71dcb115da62791a7487c6e1c9f4b6f145bd4769a87","71748.png":"6b2f48c3537fb85591ca124656a645da4bb183f2ae6a7b59ef1c20d6e9d7b5a3","740400.png":"4b73501685a7ac553dd927de36b2c0a2b14e5268138f7c4e4f87e08f651f9c16","750600.png":"b7107cc5b37b2647e73c4a82cb2ff5a8d22585555b9c9a241e5dcface1baf971","754472.png":"10d9d9e2ea6e98af52b4ea3e75b1fa82d5f9b815c7265f6463c9bbcbe80c8c75","756528.png":"712d72929be4a6fb494ad10fd3997b56e216cae1fe21723c8343fb19757b8c3e","757654.png":"5e347a388d04ec3913cdde3a74a3c52ddac1988757a71cdcf8392bbcb0154c61","761669.png":"de9c059f060b31555d9163542af3baa73a7b2d9ddff4fb2c883cc9cae56c754b","76552.png":"9e8be722fc32c09af0de2e82068a223aca506896036bcce0f9e8635a3029fd26","76645.png":"6f422429d6fe806eaedbc2bb90a872136e68dcbbf7a09f20b546fdc0280032a9","773162.png":"8fc7628db22d295679134566a8e70055918d7454e36c0019b1a74d75bc704a7d","773806.png":"ee13c8d14ebaa68c95d2b5f93fe2c422276e61edd35677f6564c9f585e4774e1","774596.png":"c9213bebf9ba1ea4476d9744cddb5fc1c65ae898285e4a530a2d06e2c1ca2899","781746.png":"867bd36f36fa6fc0fff9d5bfce03ae1db4d57b89f08b6e9c98515e0d60f30497","785007.png":"84c2bd8fb4dba5baeee830bf84894b3d0ab9bcec3f241ea9c25bc65c1f75a125","785732.png":"3f9752c8d4abbfe7def53f2e9ab06dc6daab659e58a6f1b4318abba4dddfb496","789065.png":"a389eac3a150f15963afd2b8b8759a11f4d434544b96c8699561ca3609596698","794461.png":"cdcab6d7bcdefa23de4f689fc621d4e13e9583bf7dc1cb1815d32bd0724f7510","795287.png":"94f5f32cc3669967bc4558339bc06dd9f0a3ffd1b69371c6061cdca569a9afeb","80005.png":"e2da9e547873e87e59136ec193eed7f865585625c7082741fa68095eb682bac5","802597.png":"1ed7b896a6d55ad09a0d4d07ad74af5008512fd33f6405a1d31a1d591c14b9f7","810331.png":"e1cb220074769429052472225c45bcc8ff665c7d3a7ee18ed723fd72bd4dd05b","818164.png":"975e9346f5cba67995365568616daca377e5355c76dd1815dc9fad419bdcc135","818557.png":"2b259fdfa5003c7848e77eb42dd2fa7f0d52f28048ae18af6a7b0da5b22570a0","824427.png":"aa5a8b8cbf277603463b1aee9f2ec052d9b9cfae7959229813a8a56ad984a28c","827304.png":"ba414f9e636fb915eb953a74d8ab47fae723b63af6082ddafaebb8ac4b21c8d8","829240.png":"d84bcdde0864e2c1ae9625913189d4c82055a5dd9ac9094360fa4d5a20162a40","829959.png":"ef60ccb0e39f7d35a5ebe6680834f24d1f6e2e4f87d086a522128be935ca2525","829978.png":"d4e8c5349b69bb3437cf477371ccc0f5274ac55bde769112f5c174b760bbc992","83149.png":"48db558443cdda1b161fc2677de4c18d7d49ef68719799d9da4be507fd77a8ba","84796.png":"6f688accc4c8b7319e17b3226262151bf92daa785757634ca37042369e3202f8","848486.png":"468bf964904d12142d1507760c83b11a6c502fc55084b9309cb35dae697082f8","85483.png":"50f471bcc5758da1631c7683bc85231f68aa258917a5ced12da1b3a6094e478f","858005.png":"255bb8726824066caf13900ec7e86cd236d273e4f83332e3a4def85323472325","858355.png":"37723422a126977af142b04805fe46ca18e362c9b3d5e63dd3554cc53523cde9","859158.png":"852c3a77bbfa5eb707cfa283afb34cdd11e23dcaffacbeb4cb6d41771333fcf7","861398.png":"edc1f5c2b8ee67cdb22468b9409e034f1760b6b86d7785017a91f02a409bd02f","86218.png":"9ffdb4662b3026b00d831f56d89ee609d8c6f0002e5bd1780f581dfdfd3ead45","864056.png":"d28bbab6f9d1a074a1165ab899c251a684ebcd7703ae42babb935f7eac8a5b58","86480.png":"3e362f44bd0b194ce98ff98266b21481daed3d8ac8921423c2776f04357e4167","866078.png":"435b82651e8a4237c6f7062c92d16dec6b22dc3b5acf1bfdc06c2913bc1a7bf2","875190.png":"bc897c07be9c283c4565d47ed6708702bcc9dd3afb334137d94057639193d356","881888.png":"a8e8ae3b622d02419c9dfaea8a8b6b77e1dbf1ccb55c152e79bb7876d054d231","893031.png":"ed352be63d9949295535908a696e4a79f16b7fb2d6a5c741b0c7e9155d1e3260","89437.png":"173376a275e450b1dcc1c2932210ebf155539d6e23919ac2e6cc711adbfb6361","911150.png":"3fb63dcd25fb9d23e0ebec4eb2a770f1fd0c50090c1ad148837fc4e8b9ef7fb8","912687.png":"c51d53197cd644d52dd8c05c09e25c29a70bc96bb5b252664806fd09b74c338b","913429.png":"97673ae8a3ba82fa5e4c636e69a3855258b640e8cccf493ced96cd0f985da257","91587.png":"69d1da9008ad15517b1fafdbfe350dcb5c338be7f8bf3afda3caac79c10a2a5b","916637.png":"e432af0b718cbd7745b244521427f7f48cf1823f87e8180986437a5c58f02216","919010.png":"e6d240c91e3e1e2eadd508911fb2c74689d8462661e0d61b54e1f21f0cdc72a1","929272.png":"cec6b87b7ce7a427fcde83ba1dccb2d046a419b927b0ed9158c772b3e142e855","929878.png":"0f4e96d2c90b30d3b50193631ef75144736eda76d7dd68c5bbd3b709dbbde7de","929978.png":"2b9c79ce5756551e7996bf7b5eb1f12868241cd6bcfd2827687225cde59802fa","931597.png":"4ee99379e02e35b3144dbfce33158fda9463ccf2bf25d7392b64397357fb0b5f","933281.png":"2f60ab3b534dd46618430c156662854056aed0112020d7689bbc04f342a8af40","934870.png":"406c2fb4b03d1ffcb5f0f4286eb2f2afa05c8318188722440b38430d491143ef","935810.png":"dac31b5c5644b4bfd19e3b7de01db802a4440d1e08d049865f251fcc7e0660cb","938125.png":"e1ad26b7e5c5be1ad9e263acf0f5fa5a62ab40197d0c4c40722c422a7f583797","938473.png":"3ff79f682c1612e6af5d27a2c0c4efa2fcb31754651481b23eef1fa7ef83f7cb","941351.png":"652086f6077988f10d9ab51d2255b6c82e1689e25f605309c2dffacaad995c8f","942845.png":"4f5665eb8550bdbed43fe64dab1b9c27f2245a95c6d5bf2941a3ac1e0ac9fce3","94361.png":"3eb77f4757a5bd81e26c6a5309e9ae02150329829b26a9a3f873981af6bb61e3","944502.png":"96cad7899d1608b45fbb208904eabdda1e1c0daa34beea8aad59bae3310d7143","945271.png":"3eade4c667aa3375e5d4553c90a29dbe881f8e9fd5e3fbde982a2b03b86e2b02","948729.png":"a4615ab96d9de25ef80b5c7f8340fbd97731a8fec4746ec17fcaa99420ec63f5","962847.png":"8bc2da7266e3983f0b35d744b22b805fac1b41beeac262277b10b7a46cefc1ae","964462.png":"6c6c53a620dae2872a065ad74cdce7663240a1f073a1abda054937439d15c04c","966551.png":"a3f5a7a8cbcd02275bc374cd519815adba0fd6062037ae1bc2db13acb12979be","966699.png":"0cd28e419176675082bc10fa0912373df339a245441787e1dc2645c82223a407","96883.png":"7238c021c54c2b5586702c651825a829904e9c55624d21c8fe8ce06e9d792720","972582.png":"3b7eb28e3800a1d1f24e1987d981216e3ea093d809695d70b2b733122587024c","982198.png":"93ad8f4004f2fa2f61eb942fc2f1ca85c10e2c1ed86098a767147fd7b8da5b2d","98292.png":"5ed6a77fc1de29d2f8507b6a7e7d31d01f0c69d6818e28fb26042bab1ef8d44a","99206.png":"d34bbcc69d52cae8554eb1c1f90df9e3dbd98a91fa80eb0a3719bf559073fe1c","997377.png":"4f71a6524eec84d7fc0004736b068063d770825efcb34ca7d160fc9d30b5fe82"}}''')
assert len(TEST["dhash16"]) == 250 and len(TEST["sha256"]) == 250
print(len(TEST["dhash16"]), "test frames will be kept out of training")

In [ ]:
# ---- Helpers ----
import hashlib, random, re, threading
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
import numpy as np, cv2, requests

REPO = "iisc-aim/UVH-26"
_local = threading.local()

def _session():
    if not hasattr(_local, "s"):
        _local.s = requests.Session()
        if HF_TOKEN:
            _local.s.headers["Authorization"] = f"Bearer {HF_TOKEN}"
    return _local.s

def hf_url(path, dataset=True):
    return f"https://huggingface.co/{'datasets/' if dataset else ''}{REPO}/resolve/main/{path}"

def fetch_bytes(path, size=None, dataset=True, tries=10):
    """Bytes of one repository file; waits out rate limits, retries network errors,
    and rejects a short download. `None` for a file that is not there."""
    delay = 5
    for _ in range(tries):
        try:
            r = _session().get(hf_url(path, dataset), timeout=180)
            if r.status_code == 200 and (size is None or len(r.content) == size):
                return r.content
            if r.status_code == 404:
                return None
            if r.status_code == 429:
                time.sleep(min(int(r.headers.get("Retry-After", delay)), 300))
        except requests.RequestException:
            pass
        time.sleep(delay)
        delay = min(delay * 2, 300)
    raise RuntimeError(f"could not download {path} after {tries} tries")

def fetch_file(path, dest, size=None, dataset=True):
    if os.path.exists(dest) and (size is None or os.path.getsize(dest) == size):
        return dest
    data = fetch_bytes(path, size, dataset)
    if data is None:
        raise FileNotFoundError(path)
    tmp = dest + ".part"
    with open(tmp, "wb") as f:
        f.write(data)
    os.replace(tmp, dest)
    return dest

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def dhash16(img):
    """256-bit difference hash: the fingerprint the test frames were given."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    s = cv2.resize(g, (17, 16), interpolation=cv2.INTER_AREA)
    return (s[:, 1:] > s[:, :-1]).flatten()

# IISc's 14 labels -> the six types (the same map as vision_service.classes)
UVH_TO_SIX = {"hatchback": "car", "sedan": "car", "suv": "car", "muv": "car", "van": "car",
              "bus": "bus", "mini-bus": "bus", "tempo-traveller": "bus",
              "truck": "truck", "lcv": "truck", "three-wheeler": "auto_rickshaw",
              "two-wheeler": "motorcycle", "bicycle": "bicycle", "others": None}

def six_of(name):
    key = name.strip().lower()
    if key not in UVH_TO_SIX:
        raise ValueError(f"unknown label {name!r}: stop and add it to UVH_TO_SIX")
    return UVH_TO_SIX[key]

def _iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def yolo_lines(ann_list, W, H, cat_six):
    """COCO boxes of one image -> YOLO label lines, with what was dropped and why."""
    lines, st, kept = [], Counter(), []
    for a in ann_list:
        cls = cat_six[a["category_id"]]
        if cls is None:
            st["'Others' box dropped"] += 1
            continue
        if a.get("iscrowd", 0):
            st["crowd box dropped"] += 1
            continue
        x, y, w, h = (float(v) for v in a["bbox"])
        x1, y1, x2, y2 = max(0.0, x), max(0.0, y), min(float(W), x + w), min(float(H), y + h)
        if x2 - x1 < 2 or y2 - y1 < 2:
            st["box empty, tiny or outside the image"] += 1
            continue
        if (x1, y1, x2, y2) != (x, y, x + w, y + h):
            st["box clipped to the image"] += 1
        k = SIX.index(cls)
        if any(k == k2 and _iou((x1, y1, x2, y2), b2) > 0.95 for k2, b2 in kept):
            st["duplicate box dropped"] += 1
            continue
        kept.append((k, (x1, y1, x2, y2)))
        lines.append(f"{k} {(x1 + x2) / 2 / W:.6f} {(y1 + y2) / 2 / H:.6f} "
                     f"{(x2 - x1) / W:.6f} {(y2 - y1) / H:.6f}")
        st[f"boxes: {cls}"] += 1
    return lines, st

# ---- BatchNorm: the part version 1 lacked ----
FP16_MAX = 65504.0

def bn_report(model):
    """BatchNorm health: channels with non-finite statistics, and channels whose variance is too large for a
    half-precision file (such a file stores them as infinity, or clipped to 65,504)."""
    bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]
    rv = torch.cat([m.running_var.detach().float().flatten().cpu() for m in bns])
    rm = torch.cat([m.running_mean.detach().float().flatten().cpu() for m in bns])
    bad = ~(torch.isfinite(rv) & torch.isfinite(rm))
    return {"bn_layers": len(bns), "channels": int(rv.numel()), "nonfinite_channels": int(bad.sum()),
            "channels_over_fp16_max": int((rv[~bad] > FP16_MAX).sum()),
            "channels_at_fp16_max": int((rv[~bad] == FP16_MAX).sum()),
            "max_running_var": float(rv[~bad].max()) if (~bad).any() else None}

AUG_KEYS = ("mosaic", "mixup", "cutmix", "copy_paste", "degrees", "translate", "scale", "shear", "perspective",
            "flipud", "fliplr", "hsv_h", "hsv_s", "hsv_v", "bgr")

def bn_recalibrate(model, data_yaml, size, train_args, n_batches=BN_BATCHES, batch=16, workers=None, seed=SEED):
    """Re-estimate every BatchNorm layer's statistics in full precision, as a cumulative average over
    `n_batches` training batches augmented the way the model was trained (`train_args`), at input `size`.
    Only the statistics change; no learned weight is touched. Training images only, never the test frames."""
    from ultralytics.cfg import get_cfg
    from ultralytics.data import build_dataloader, build_yolo_dataset
    from ultralytics.data.utils import check_det_dataset
    from ultralytics.utils.torch_utils import init_seeds

    init_seeds(seed, deterministic=True)       # the same augmented images on every run
    aug = {k: train_args[k] for k in AUG_KEYS if k in train_args}
    cfg = get_cfg(overrides={**aug, "imgsz": size, "rect": False, "cache": False, "fraction": 1.0,
                             "seed": seed, "deterministic": True})
    data = check_det_dataset(data_yaml)
    ds = build_yolo_dataset(cfg, data["train"], batch, data, mode="train", stride=32)
    dl = build_dataloader(ds, batch, BN_WORKERS if workers is None else workers, shuffle=True, rank=-1)
    dev = next(model.parameters()).device
    model.float()
    bns = [m for m in model.modules() if isinstance(m, torch.nn.BatchNorm2d)]
    for m in bns:
        m.reset_running_stats()
        m.momentum = None                     # cumulative average, not an exponential one
    model.train()
    seen = 0
    try:
        with torch.no_grad(), torch.autocast(dev.type, enabled=False):
            for i, b in enumerate(dl):
                if i >= n_batches:
                    break
                model(b["img"].to(dev).float() / 255)
                seen += b["img"].shape[0]
    finally:
        model.eval()
        for m in bns:
            m.momentum = 0.03                 # Ultralytics' value
        if hasattr(dl, "close"):
            dl.close()
    rep = bn_report(model)
    assert seen > 0 and rep["nonfinite_channels"] == 0, rep
    return {"images": seen, "imgsz": size, "augment": aug, **rep}

def save_fp32(model, path, train_args, **extra):
    """An Ultralytics-loadable checkpoint that keeps every number in full precision (Ultralytics' own saves are
    half precision). It carries no optimizer state, so it can never be mistaken for a run to resume."""
    ck = {"epoch": -1, "best_fitness": None, "model": deepcopy(model).float().cpu(), "ema": None,
          "optimizer": None, "train_args": dict(train_args), "date": time.strftime("%Y-%m-%dT%H:%M:%S"), **extra}
    tmp = path + ".part"
    torch.save(ck, tmp)
    os.replace(tmp, path)
    return path

def place(src, dst):
    """Copy unless it is already there (an export lands next to the model it came from)."""
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    return dst

def letterbox_rect(img, size, stride=32):
    """The letterbox the ITMS detectors use: fit into `size`, pad only up to a multiple of `stride`."""
    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    ch, cw = nh + (size - nh) % stride, nw + (size - nw) % stride
    canvas = np.full((ch, cw, 3), 114, np.uint8)
    top, left = (ch - nh) // 2, (cw - nw) // 2
    canvas[top:top + nh, left:left + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas

def verify_onnx(onnx_path, torch_model, images, size):
    """The exported file must give the numbers the full-precision model gives, on real frames at the
    letterbox the detectors use. Returns the worst differences."""
    import onnxruntime as ort
    s = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    net = deepcopy(torch_model).float().cpu().eval()
    worst_box, worst_score, n_out = 0.0, 0.0, None
    for p in images:
        x = letterbox_rect(cv2.imread(p), size)
        x = np.ascontiguousarray(x[:, :, ::-1].transpose(2, 0, 1)[None], dtype=np.float32) / 255.0
        a = s.run(None, {s.get_inputs()[0].name: x})[0]
        with torch.no_grad():
            b = net(torch.from_numpy(x))
        b = (b[0] if isinstance(b, (list, tuple)) else b).numpy()
        assert a.shape == b.shape, (a.shape, b.shape)
        assert np.isfinite(a).all(), "the exported model gives non-finite numbers"
        n_out = a.shape[1]
        worst_box = max(worst_box, float(np.abs(a[:, :4] - b[:, :4]).max()))
        worst_score = max(worst_score, float(np.abs(a[:, 4:] - b[:, 4:]).max()))
    assert worst_box < 0.05 and worst_score < 1e-3, ("the exported model disagrees with the original", worst_box,
                                                   worst_score)
    return {"frames": len(images), "outputs": n_out, "max_box_diff_px": round(worst_box, 5),
            "max_score_diff": float(f"{worst_score:.2e}")}

In [ ]:
# ---- Labels (needed for the class head's starting rows, and to build the dataset if Drive has none) ----
lab_path = fetch_file("UVH-26-Train/UVH-26-MV-Train.json", f"{WORK}/UVH-26-MV-Train.json")
coco = json.load(open(lab_path))
cat_six = {c["id"]: six_of(c["name"]) for c in coco["categories"]}   # stops on an unknown label
print("labels:", {c["name"]: cat_six[c["id"]] for c in coco["categories"]})
anns = defaultdict(list)
for a in coco["annotations"]:
    anns[a["image_id"]].append(a)
# each of the six types starts from the head row of its most common IISc label (e.g. two-wheeler <- Two-wheeler)
_n = Counter(a["category_id"] for a in coco["annotations"])
REP = {t: max((c for c in coco["categories"] if cat_six[c["id"]] == t), key=lambda c: _n[c["id"]])["name"]
       for t in SIX}
print("head rows start from:", REP)
images = {}
for im in coco["images"]:
    n = os.path.basename(im["file_name"])
    assert n not in images, f"two labelled images share the name {n}"
    images[n] = im

In [ ]:
# ---- The dataset: restored from Drive (prepared by version 1), or built ----
DS = f"{WORK}/ds"
MANIFEST = f"{OUT}/dataset_manifest.json"
TAR = f"{OUT}/dataset.tar"
SETTINGS = {"N_TRAIN": N_TRAIN, "N_HOLDOUT": N_HOLDOUT, "SEED": SEED, "SAME_CAMERA_BITS": SAME_CAMERA_BITS,
            "MAX_SIDE": MAX_SIDE}
test_bits = np.array([np.unpackbits(np.frombuffer(bytes.fromhex(h), np.uint8)) for h in TEST["dhash16"].values()])
test_sha = set(TEST["sha256"].values())

def _size(f):
    lfs = getattr(f, "lfs", None)
    s = lfs.get("size") if isinstance(lfs, dict) else getattr(lfs, "size", None)
    return s or getattr(f, "size", None)

def list_pool():
    from huggingface_hub import HfApi
    files = {}
    for f in HfApi(token=HF_TOKEN).list_repo_tree(REPO, repo_type="dataset", path_in_repo="UVH-26-Train/data",
                                                   recursive=True):
        if f.path.lower().endswith((".png", ".jpg", ".jpeg")):
            files[os.path.basename(f.path)] = (f.path, _size(f))
    test_names = set(TEST["sha256"])
    pool = sorted((set(images) & set(files)) - test_names)
    print(f"{len(images)} labelled training images, {len(files)} image files, {len(pool)} usable "
          f"({len(set(images) - set(files))} labelled but missing, {len(set(files) - set(images))} unlabelled, "
          f"{len((set(images) & set(files)) & test_names)} named like a test frame)")
    assert len(pool) >= N_TRAIN + N_HOLDOUT, "not enough images for the requested sample"
    return files, pool

def process(name, data):
    """(reason if skipped, (image, label lines, box stats, fingerprint distance))"""
    if data is None:
        return "file missing", None
    if hashlib.sha256(data).hexdigest() in test_sha:
        return "identical to a test frame", None
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        return "does not open", None
    meta = images[name]
    H, W = img.shape[:2]
    if (W, H) != (meta["width"], meta["height"]):
        return "size differs from the label file", None
    dist = int((test_bits != dhash16(img).astype(np.uint8)).sum(1).min())
    if dist <= SAME_CAMERA_BITS:
        return "looks like a test camera", None
    lines, st = yolo_lines(anns[meta["id"]], W, H, cat_six)
    return None, (img, lines, st, dist)

def build():
    files, pool = list_pool()
    for d in ("images/train", "images/val", "labels/train", "labels/val"):
        os.makedirs(f"{DS}/{d}", exist_ok=True)
    order = pool[:]
    random.Random(SEED).shuffle(order)
    need = N_TRAIN + N_HOLDOUT
    accepted, skipped, box_stats = [], defaultdict(list), Counter()
    t0, i = time.time(), 0
    with ThreadPoolExecutor(8) as ex:
        while len(accepted) < need and i < len(order):
            chunk = order[i:i + 48]
            i += len(chunk)
            got = ex.map(lambda n: (n, fetch_bytes(*files[n])), chunk)   # order preserved: reproducible
            for name, data in got:
                if len(accepted) >= need:
                    break
                why, res = process(name, data)
                if why:
                    skipped[why].append(name)
                    continue
                img, lines, st, _ = res
                split = "val" if len(accepted) < N_HOLDOUT else "train"
                stem = os.path.splitext(name)[0]
                h, w = img.shape[:2]
                if max(h, w) > MAX_SIDE:
                    s = MAX_SIDE / max(h, w)
                    img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
                assert cv2.imwrite(f"{DS}/images/{split}/{stem}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])
                with open(f"{DS}/labels/{split}/{stem}.txt", "w") as f:
                    f.write("\n".join(lines) + ("\n" if lines else ""))
                accepted.append((name, split, len(lines)))
                box_stats.update(st)
            print(f"  {len(accepted)}/{need} kept, {sum(map(len, skipped.values()))} skipped, "
                  f"{time.time() - t0:.0f} s", flush=True)
    assert len(accepted) == need, f"only {len(accepted)} usable images found"
    counts = {sp: Counter() for sp in ("train", "val")}
    for sp in counts:
        for p in os.listdir(f"{DS}/labels/{sp}"):
            for line in open(f"{DS}/labels/{sp}/{p}"):
                counts[sp][SIX[int(line.split()[0])]] += 1
    man = {"settings": dict(SETTINGS),
           "labels_file": "UVH-26-Train/UVH-26-MV-Train.json", "labels_sha256": sha256_file(lab_path),
           "train": [n for n, s, _ in accepted if s == "train"], "holdout": [n for n, s, _ in accepted if s == "val"],
           "images_without_boxes": sum(1 for *_, k in accepted if k == 0),
           "skipped": {k: v for k, v in skipped.items()}, "box_stats": dict(box_stats),
           "vehicles_by_type": {sp: dict(c) for sp, c in counts.items()}}
    json.dump(man, open(MANIFEST, "w"), indent=1)
    subprocess.run(["tar", "-cf", TAR + ".part", "-C", WORK, "ds"], check=True)
    os.replace(TAR + ".part", TAR)
    return man

if os.path.exists(MANIFEST) and os.path.exists(TAR):
    man = json.load(open(MANIFEST))
    assert man["settings"] == SETTINGS, (
        f"The dataset in Drive was made with {man['settings']}, these settings are {SETTINGS}. Put the settings "
        "back, or move dataset.tar and dataset_manifest.json out of the Drive folder to build a new one.")
    if man["labels_sha256"] != sha256_file(lab_path):
        print("NOTE: IISc's label file changed since the dataset was built; the dataset keeps the labels it was "
              "built with (only the class head's starting rows are read from the new file)")
    if not os.path.isdir(DS):
        print("restoring the prepared dataset from Drive...")
        subprocess.run(["tar", "-xf", TAR, "-C", WORK], check=True)
else:
    man = build()
for sp, key in (("train", "train"), ("val", "holdout")):
    stems = {os.path.splitext(n)[0] for n in man[key]}
    imgs = {os.path.splitext(n)[0] for n in os.listdir(f"{DS}/images/{sp}") if n.endswith(".jpg")}
    labs = {os.path.splitext(n)[0] for n in os.listdir(f"{DS}/labels/{sp}") if n.endswith(".txt")}
    assert stems == imgs == labs, f"the {sp} split on disk does not match the manifest"
assert not {os.path.splitext(n)[0] for n in TEST["sha256"]} & {os.path.splitext(n)[0] for n in man["train"] + man["holdout"]}
print("kept:", len(man["train"]), "train +", len(man["holdout"]), "hold-out")
print("skipped:", {k: len(v) for k, v in man["skipped"].items()})
print("boxes:", man["box_stats"])
print("vehicles by type:", man["vehicles_by_type"])
for sp in ("train", "val"):
    missing = [c for c in SIX if not man["vehicles_by_type"][sp].get(c)]
    if missing:
        print(f"WARNING: no {missing} in the {sp} split")

In [ ]:
# ---- Look at the labels: boxes drawn on six training images (check they sit on the vehicles) ----
from IPython.display import Image as _Img, display
_cols = [(0, 200, 255), (0, 255, 0), (255, 128, 0), (255, 0, 255), (0, 0, 255), (255, 255, 0)]
tiles = []
for name in random.Random(SEED + 1).sample(man["train"], min(6, len(man["train"]))):
    stem = os.path.splitext(name)[0]
    img = cv2.imread(f"{DS}/images/train/{stem}.jpg")
    h, w = img.shape[:2]
    for line in open(f"{DS}/labels/train/{stem}.txt"):
        k, cx, cy, bw, bh = line.split()
        k, cx, cy, bw, bh = int(k), float(cx) * w, float(cy) * h, float(bw) * w, float(bh) * h
        p1, p2 = (int(cx - bw / 2), int(cy - bh / 2)), (int(cx + bw / 2), int(cy + bh / 2))
        cv2.rectangle(img, p1, p2, _cols[k], 2)
        cv2.putText(img, SIX[k], (p1[0], max(12, p1[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, _cols[k], 1)
    tiles.append(cv2.resize(img, (640, 360)))
tiles += [np.zeros_like(tiles[0])] * (len(tiles) % 2)
sheet = np.vstack([np.hstack(tiles[i:i + 2]) for i in range(0, len(tiles), 2)])
cv2.imwrite(f"{OUT}/label_check.jpg", sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])
display(_Img(f"{OUT}/label_check.jpg"))

In [ ]:
# ---- IISc's model and the dataset description ----
w_path = fetch_file(IISC_WEIGHTS, f"{WORK}/UVH-26-MV-YOLOv11-S.pt", size=IISC_BYTES, dataset=False)
assert sha256_file(w_path) == IISC_SHA256, "IISc's model file changed since it was measured; stop and check"
IISC_ARGS = torch.load(w_path, map_location="cpu", weights_only=False)["train_args"]
YAML = f"{WORK}/itms_six.yaml"
with open(YAML, "w") as f:
    f.write(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n")
    f.write("".join(f"  {i}: {n}\n" for i, n in enumerate(SIX)))
print(open(YAML).read())
HOLDOUT_PROBE = sorted(f"{DS}/images/val/{n}" for n in os.listdir(f"{DS}/images/val"))[:4]  # for export checks
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
REPORT_PATH = f"{OUT}/train_report_v2.json"
report = json.load(open(REPORT_PATH)) if os.path.exists(REPORT_PATH) else {"models": {}, "controls": {}}
report.update({"ultralytics": ULTRALYTICS_VERSION, "torch": torch.__version__,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
               "iisc_sha256": IISC_SHA256, "seed": SEED,
               "settings": {"EPOCHS": EPOCHS, "PATIENCE": PATIENCE, "BATCH": BATCH, "BN_BATCHES": BN_BATCHES}})

def save_report():
    tmp = REPORT_PATH + ".part"
    json.dump(report, open(tmp, "w"), indent=1)
    os.replace(tmp, REPORT_PATH)

In [ ]:
# ---- Training and export: the steps ----
from ultralytics import YOLO
from ultralytics.utils.torch_utils import unwrap_model
import zipfile
RUNS = f"{OUT}/runs"

def warm_start_head(trainer):
    """The new 6-way class layer starts from IISc's rows for the matching labels instead of random numbers
    (Ultralytics matches rows by name only, which covers bus, truck and bicycle but not car, auto-rickshaw or
    two-wheeler). Runs once, before the first epoch; never on a resumed run. (The rows have no BatchNorm.)"""
    src = torch.load(w_path, map_location="cpu", weights_only=False)["ema"].float()
    src_idx = {str(v).strip().lower(): k for k, v in src.names.items()}
    rows = [src_idx[REP[t].strip().lower()] for t in SIX]
    ssd = src.state_dict()
    for mod in (trainer.model, getattr(trainer.ema, "ema", None)):
        if mod is None:
            continue
        mod = unwrap_model(mod)
        sd = mod.state_dict()
        keys = [k for k in sd if re.fullmatch(r"model\.\d+\.cv3\.\d+\.2\.(weight|bias)", k)]
        assert len(keys) == 6, keys
        with torch.no_grad():
            for k in keys:
                assert sd[k].shape[0] == len(SIX) and ssd[k].shape[1:] == sd[k].shape[1:], k
                sd[k].copy_(ssd[k][rows].to(sd[k].device, sd[k].dtype))
    print("class head started from IISc rows:", dict(zip(SIX, (REP[t] for t in SIX))))

def check_start(trainer):
    """Before the first epoch: full precision, and no broken BatchNorm statistics in the model being trained."""
    assert not trainer.amp, "mixed precision is on; this run must train in full precision"
    for mod in (trainer.model, trainer.ema.ema):
        rep = bn_report(unwrap_model(mod))
        assert rep["nonfinite_channels"] == 0, ("training would start from broken statistics", rep)

def nan_guard(trainer):
    """In full precision a non-finite loss is not an overflow to skip past: it poisons the weights. Stop."""
    if trainer.loss is not None and not torch.isfinite(trainer.loss).all():
        raise RuntimeError(
            f"A training batch gave a non-finite loss (epoch {trainer.epoch + 1}). Stop here and send Claude this "
            "output: running again would only repeat it.")

def save_best_fp32(trainer):
    """Keep a full-precision copy of the best model: Ultralytics' best.pt is half precision, which cannot hold
    the larger BatchNorm variances."""
    if trainer.best_fitness is not None and trainer.fitness == trainer.best_fitness:
        save_fp32(unwrap_model(trainer.ema.ema), str(trainer.wdir / "best_fp32.pt"), vars(trainer.args),
                  trained_epoch=trainer.epoch + 1, best_fitness=float(trainer.best_fitness))

def save_last_fp32(trainer):
    """Every epoch, beside last.pt: the same model in full precision, so that a resume is exact."""
    sd = {k: v.detach().cpu().clone().float() if v.is_floating_point() else v.detach().cpu().clone()
          for k, v in unwrap_model(trainer.ema.ema).state_dict().items()}
    path = str(trainer.wdir / "last_fp32.pt")
    torch.save({"epoch": trainer.epoch, "ema": sd}, path + ".part")
    os.replace(path + ".part", path)

RESUMES = []

def restore_fp32(trainer):
    """On a resume Ultralytics restarts from last.pt, whose half precision has lost the large BatchNorm variances
    (they come back as 65,504). Put back the full-precision copy saved with it, into the model and its average."""
    p = trainer.wdir / "last_fp32.pt"
    ck = torch.load(p, map_location="cpu", weights_only=False) if p.exists() else None
    if ck is None or ck["epoch"] != trainer.start_epoch - 1:
        RESUMES.append({"from_epoch": trainer.start_epoch, "full_precision_restored": False})
        print("WARNING: no full-precision copy for this epoch; the clipped statistics re-converge during training "
              "(the result is still usable; tell Claude)")
        return
    for mod in (trainer.model, trainer.ema.ema):
        m = unwrap_model(mod)
        dev = next(m.parameters()).device
        m.load_state_dict({k: v.to(dev) for k, v in ck["ema"].items()})
    RESUMES.append({"from_epoch": trainer.start_epoch, "full_precision_restored": True})
    print("resumed with the full-precision copy of epoch", ck["epoch"] + 1)

FRESH_CALLBACKS = [("on_pretrain_routine_end", warm_start_head), ("on_pretrain_routine_end", check_start)]
RESUME_CALLBACKS = [("on_pretrain_routine_end", restore_fp32)]
CALLBACKS = [("on_train_batch_end", nan_guard), ("on_model_save", save_best_fp32), ("on_model_save", save_last_fp32)]

def control(size):
    """IISc's model with nothing changed but its BatchNorm statistics, re-estimated in full precision on our
    training images, augmented as IISc trained (their train_args), at this input size. Also the start of training."""
    pt, onnx_out = f"{OUT}/iisc_bnfix_{size}.pt", f"{OUT}/iisc_bnfix_{size}.onnx"
    key = str(size)
    if key in report["controls"] and os.path.exists(pt) and os.path.exists(onnx_out):
        return pt
    yolo = YOLO(w_path)
    before = bn_report(yolo.model)
    yolo.model.to(DEV)
    info = bn_recalibrate(yolo.model, YAML, size, IISC_ARGS, batch=16)
    save_fp32(yolo.model, pt, IISC_ARGS, bn_recalibrated=info)
    fixed = YOLO(pt)
    assert bn_report(fixed.model)["nonfinite_channels"] == 0
    exp = fixed.export(format="onnx", dynamic=True, imgsz=size, half=False, simplify=True, device="cpu")
    place(exp, onnx_out)
    check = verify_onnx(onnx_out, fixed.model, HOLDOUT_PROBE, size)
    report["controls"][key] = {"bn_before": before, "bn_recalibration": info, "onnx_check": check,
                               "onnx_sha256": sha256_file(onnx_out), "pt_sha256": sha256_file(pt)}
    save_report()
    print(f"control iisc_bnfix_{size}: {before['nonfinite_channels']} broken channels before, "
          f"{info['nonfinite_channels']} after; {info['channels_over_fp16_max']} channels above half precision; "
          f"export check {check}")
    return pt

def train(size, start):
    name = f"{RUN_TAG}_{size}"
    last = f"{RUNS}/{name}/weights/last.pt"
    t0 = time.time()
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("optimizer") is None:        # a finished run keeps no optimizer state
            print(name, "already finished")
            return None
        print(name, "continuing from epoch", ck.get("epoch", -1) + 1)
        yolo = YOLO(last)
        for ev, fn in RESUME_CALLBACKS + CALLBACKS:
            yolo.add_callback(ev, fn)
        yolo.train(resume=True)
    else:
        yolo = YOLO(start)
        for ev, fn in FRESH_CALLBACKS + CALLBACKS:
            yolo.add_callback(ev, fn)
        yolo.train(
            data=YAML, imgsz=size, epochs=EPOCHS, patience=PATIENCE, batch=BATCH[size],
            optimizer="AdamW", lr0=0.001, lrf=0.1, cos_lr=True, warmup_epochs=2, weight_decay=5e-4,
            mosaic=0.5, close_mosaic=5, scale=0.3, translate=0.1, fliplr=0.5, mixup=0.0,
            seed=SEED, deterministic=True, workers=2, cache="disk", amp=False,
            project=RUNS, name=name, exist_ok=True, plots=True, save_period=-1)
    minutes = round((time.time() - t0) / 60)
    tr = yolo.trainer
    return {"minutes_this_session": minutes, "batch_used": int(tr.batch_size), "epochs_run": int(tr.epoch) + 1,
            "resumed": list(RESUMES)}

def finalize(size, run_info):
    """The best epoch, in full precision: hold-out check, ONNX export straight from it, export check."""
    name = f"{RUN_TAG}_{size}"
    wdir = f"{RUNS}/{name}/weights"
    key = str(size)
    rec = report["models"].get(key, {})
    if run_info:
        rec.setdefault("sessions", []).append(run_info)
    out_pt, out_onnx = f"{OUT}/itms_six_fp32_{size}.pt", f"{OUT}/itms_six_fp32_{size}.onnx"
    if rec.get("onnx_sha256") and os.path.exists(out_onnx) and sha256_file(out_onnx) == rec["onnx_sha256"]:
        return rec
    if os.path.exists(f"{wdir}/best_fp32.pt"):
        shutil.copy(f"{wdir}/best_fp32.pt", out_pt)
        yolo = YOLO(out_pt)
        rec["source"] = "best epoch, kept in full precision during training"
        rec["best_epoch"] = yolo.ckpt.get("trained_epoch")
    else:   # should not happen: rebuild the statistics the half-precision best.pt lost
        yolo = YOLO(f"{wdir}/best.pt")
        yolo.model.to(DEV)
        rec["source"] = "best.pt with its BatchNorm statistics re-estimated (no full-precision copy was found)"
        rec["bn_recalibration"] = bn_recalibrate(yolo.model, YAML, size, yolo.ckpt["train_args"], batch=16)
        save_fp32(yolo.model, out_pt, yolo.ckpt["train_args"], bn_recalibrated=rec["bn_recalibration"])
        yolo = YOLO(out_pt)
    assert list(yolo.names.values()) == SIX, yolo.names
    rec["bn"] = bn_report(yolo.model)
    assert rec["bn"]["nonfinite_channels"] == 0, rec["bn"]
    exp = yolo.export(format="onnx", dynamic=True, imgsz=size, half=False, simplify=True, device="cpu")
    place(exp, out_onnx)
    rec["onnx_check"] = verify_onnx(out_onnx, yolo.model, HOLDOUT_PROBE, size)
    r = YOLO(out_pt).val(data=YAML, imgsz=size, batch=16, split="val", half=False, plots=False, verbose=False)
    rec.update({"holdout_map50": round(float(r.box.map50), 4), "holdout_map50_95": round(float(r.box.map), 4),
                "holdout_ap50_by_type": {SIX[int(c)]: round(float(r.box.ap50[i]), 4)
                                         for i, c in enumerate(r.box.ap_class_index)},
                "pt_sha256": sha256_file(out_pt), "onnx_sha256": sha256_file(out_onnx)})
    report["models"][key] = rec
    save_report()
    print(size, {k: rec[k] for k in ("source", "best_epoch", "holdout_map50", "holdout_map50_95", "onnx_check")
                 if k in rec})
    return rec

def package():
    """Everything Claude needs, in one zip in Drive (rewritten after each model)."""
    zp = f"{OUT}/{ZIP_NAME}"
    with zipfile.ZipFile(zp + ".part", "w", zipfile.ZIP_DEFLATED) as z:
        names = ["train_report_v2.json", "dataset_manifest.json", "label_check.jpg"]
        for s in SIZES:
            names += [f"iisc_bnfix_{s}.onnx", f"itms_six_fp32_{s}.pt", f"itms_six_fp32_{s}.onnx"]
        for fn in names:
            if os.path.exists(f"{OUT}/{fn}"):
                z.write(f"{OUT}/{fn}", fn)
        for s in SIZES:
            for fn in ("results.csv", "args.yaml", "confusion_matrix.png", "results.png"):
                p = f"{RUNS}/{RUN_TAG}_{s}/{fn}"
                if os.path.exists(p):
                    z.write(p, f"runs/{RUN_TAG}_{s}/{fn}")
    os.replace(zp + ".part", zp)
    print("saved", zp, f"({os.path.getsize(zp) / 1e6:.0f} MB)")
    return zp

In [ ]:
# ---- Run: for each input size, the control, training, export, and the zip brought up to date ----
for size in SIZES:
    start = control(size)
    info = train(size, start)
    finalize(size, info)
    package()

In [ ]:
# ---- Download ----
ZIP = package()
try:
    from google.colab import files as _files
    _files.download(ZIP)
except Exception:  # noqa: BLE001
    print("Download it from Drive: folder itms_uvh26_finetune")